# 05 — External validation

NBER recession overlap and VIX>20 baseline on named Stressful labels.

**Section A:** committed CSVs (GMM pointwise; HMM global Viterbi — look-ahead).  
**Section B:** refit on full-sample std; HMM **forward-filtered** (causal) decode — the honest deployable labels.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from regime_utils import *

raw = load_raw_features()
gmm = load_regime_labels("gmm_regimes.csv")
hmm = load_regime_labels("hmm_regimes.csv")

df = raw.copy()
df["Regime_GMM"] = gmm["Regime_label"]
df["Regime_HMM"] = hmm["Regime_label"]
df["NBER_recession"] = df.index.map(in_recession)
print(f"Regime_GMM non-null: {df['Regime_GMM'].notna().sum()} / {len(df)}")
df.head()


Regime_GMM non-null: 1564 / 1564


,SP500,SP500_Return,VIX,Yield_Spread,Regime_GMM,Regime_HMM,NBER_recession
Date,,,,,,,
1994-01-14,474.910004,0.010605,11.15,1.61,Calm,Transitional,False
1994-01-21,474.720001,-0.000400,11.09,1.64,Calm,Transitional,False
1994-01-28,478.700012,0.008349,9.94,1.60,Calm,Transitional,False
1994-02-04,469.809998,-0.018746,15.25,1.52,Calm,Transitional,False
1994-02-11,470.179993,0.000787,14.46,1.44,Calm,Transitional,False


### A. Committed look-ahead labels

GMM is already pointwise (no decode step). Committed HMM labels use global Viterbi (`hmm.predict`).

In [2]:
nber_table = pd.DataFrame({
    "GMM": nber_stress_rates(df, "Regime_GMM"),
    "HMM (Viterbi)": nber_stress_rates(df, "Regime_HMM"),
})
print(nber_table.round(3))


                                       GMM      HMM
Recession weeks (% Stressful)        0.522    0.597
Non-recession weeks (% Stressful)    0.093    0.222
Recession weeks (n)                134.000  134.000


In [3]:

for start, end, name in NBER_RECESSIONS:
    sub = df.loc[start:end]
    print(f"{name}: GMM {(sub['Regime_GMM']==STRESS_LABEL).mean():.1%} Stressful, "
          f"HMM {(sub['Regime_HMM']==STRESS_LABEL).mean():.1%} Stressful")


Dot-com: GMM 30.0% Stressful, HMM 57.5% Stressful
GFC: GMM 65.9% Stressful, HMM 58.5% Stressful
COVID: GMM 33.3% Stressful, HMM 75.0% Stressful


In [4]:
baseline_table = pd.DataFrame({
    "GMM": vix_baseline_metrics(df, "Regime_GMM"),
    "HMM (Viterbi)": vix_baseline_metrics(df, "Regime_HMM"),
})
print(baseline_table.round(3))


                                      GMM    HMM
Agreement with VIX>20 (%)           0.729  0.783
Cohen's kappa (Stress vs rest)      0.350  0.513
Model Stress recall vs baseline     0.318  0.549
Model Stress precision vs baseline  0.951  0.837


### B. Causal filtered decode

Refit on the same full-sample std as notebook 03, then compare GMM (pointwise), HMM Viterbi (look-ahead), and HMM forward-filtered (`filtered_decode` — causal). Names assigned per fit via mean VIX.

In [ ]:
std = load_full_sample_std()
X = std.values
raw_fit = load_raw_features().loc[std.index]

gmm = fit_gmm(X)
hmm = fit_hmm(X)
gmm_labels = build_regime_frame(raw_fit, gmm.predict(X), "Regime_GMM")["Regime_label"]
vit_labels = build_regime_frame(raw_fit, viterbi_decode(hmm, X), "Regime_HMM")["Regime_label"]
filt_labels = build_regime_frame(raw_fit, filtered_decode(hmm, X), "Regime_HMM_filtered")["Regime_label"]

df_causal = raw_fit.copy()
df_causal["Regime_GMM"] = gmm_labels.values
df_causal["Regime_HMM_Viterbi"] = vit_labels.values
df_causal["Regime_HMM_filtered"] = filt_labels.values
df_causal["NBER_recession"] = df_causal.index.map(in_recession)

n_diff = (df_causal["Regime_HMM_Viterbi"] != df_causal["Regime_HMM_filtered"]).sum()
print(f"Weeks where Viterbi ≠ filtered: {n_diff} / {len(df_causal)} ({n_diff / len(df_causal):.1%})")

In [ ]:
nber_causal = pd.DataFrame({
    "GMM (pointwise)": nber_stress_rates(df_causal, "Regime_GMM"),
    "HMM Viterbi": nber_stress_rates(df_causal, "Regime_HMM_Viterbi"),
    "HMM filtered": nber_stress_rates(df_causal, "Regime_HMM_filtered"),
})
print(nber_causal.round(3))

In [ ]:
for start, end, name in NBER_RECESSIONS:
    sub = df_causal.loc[start:end]
    print(
        f"{name}: GMM {(sub['Regime_GMM'] == STRESS_LABEL).mean():.1%} Stressful, "
        f"Viterbi {(sub['Regime_HMM_Viterbi'] == STRESS_LABEL).mean():.1%}, "
        f"filtered {(sub['Regime_HMM_filtered'] == STRESS_LABEL).mean():.1%} Stressful"
    )

In [ ]:
vix_causal = pd.DataFrame({
    "GMM (pointwise)": vix_baseline_metrics(df_causal, "Regime_GMM"),
    "HMM Viterbi": vix_baseline_metrics(df_causal, "Regime_HMM_Viterbi"),
    "HMM filtered": vix_baseline_metrics(df_causal, "Regime_HMM_filtered"),
})
print(vix_causal.round(3))

In [ ]:
vit_stress = df_causal["Regime_HMM_Viterbi"] == STRESS_LABEL
filt_stress = df_causal["Regime_HMM_filtered"] == STRESS_LABEL
print(f"HMM Stress label agreement (Viterbi vs filtered): {(vit_stress == filt_stress).mean():.1%}")
print(f"Filtered recall vs Viterbi Stress: {(filt_stress & vit_stress).sum() / vit_stress.sum():.3f}")
print(f"Filtered precision vs Viterbi Stress: {(filt_stress & vit_stress).sum() / filt_stress.sum():.3f}")